# Transferência de Aprendizagem com VGG-16

## Etapa 1 - Importando as bibliotecas

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
import zipfile

cv2.__version__ # confere a versão do opencv instalada

In [ ]:
%tensorflow_version 2.x
import tensorflow
tensorflow.__version__ # confere a versão instalada

## Etapa 2 - Conectando com o Drive e acessando os arquivos

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# extrai o material do curso (que já foi baixado do drive)
path = "/content/gdrive/My Drive/Material.zip"
zip_object = zipfile.ZipFile(file=path, mode="r")
zip_object.extractall("./")

# extrai também o dataset fer2013, que fica zipado dentro do Material
base_imgs = 'Material/fer2013.zip'
zip_object = zipfile.ZipFile(file = base_imgs, mode = "r")
zip_object.extractall("./")
zip_object.close()

## Etapa 3 - Acessando a base com fotos de expressões faciais



In [ ]:
# diretório do drive onde estão os arquivos do curso (csv, modelos, fotos de teste etc)
diretorio = 'gdrive/My Drive/Cursos/Deteccao_Expressoes_Faciais/' # diretorio n drive onde estão os arquivos do curso

data = pd.read_csv(diretorio + 'fer2013/fer2013.csv')
data.tail() # só pra ver as últimas linhas e conferir o formato

In [ ]:
# histograma pra ver como as emoções estão distribuídas no dataset
plt.figure(figsize=(12,6))
plt.hist(data['emotion'], bins=6)
plt.title("Imagens x emoção")
plt.show()

# Classes: ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

## Etapa 4 - Pré-processamento

In [ ]:
largura, altura = 48, 48

faces = []   # vai guardar todas as imagens já convertidas em matriz
amostras = 0
for pixel_sequence in pixels:
  face = [int(pixel) for pixel in pixel_sequence.split(' ')]
  face = np.asarray(face).reshape(largura, altura, 1)
  # As imagens da nossa base estão em grayscale (escala de cinza) então precisaremos converter para RGB
  # para realizar isso vamos replicar o valor grayscale para cada um dos 3 canais, por exemplo: antes=(255) e depois=(255,255,255)
  # isso pode ser feito com o comando abaixo:
  face = np.asarray(np.dstack((face, face, face)), dtype=np.uint8)
  # o shape será alterado de (largura, altura) para (largura, altura, 3)
  # precisa deixar com 3 canais de cor, pois a rede VGG-16 foi treinada com imagens coloridas, então ela aceita como input imagens de 3 dimensões (3 canais de cor)
  faces.append(face)

  if (amostras < 10):
    cv2_imshow(face)

  amostras = amostras + 1

faces = np.asarray(faces)

# normaliza os pixels: de 0-255 pra 0-1 (ajuda a rede a convergir mais rápido)
def normalizar(x):
    x = x.astype('float32')
    x = x / 255.0
    return x

faces = normalizar(faces)

# transforma o id da emoção (0 a 6) em one-hot encoding, formato que a rede espera na saída
emocoes = pd.get_dummies(data['emotion']).as_matrix()

In [ ]:
# confere quantas imagens no total foram carregadas
print("Número total de imagens no dataset: "+str(len(faces)))

## Etapa 5 - Imports do Tensorflow/Keras

In [ ]:
from sklearn.model_selection import train_test_split# divide os dados em treino/teste/validação

# camadas e utilitários do keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# callbacks: controlam o treinamento (reduzir learning rate, parar cedo, salvar o melhor modelo)
from tensorflow.keras.callbacks import ReduceLROnPlateau, TensorBoard, EarlyStopping, ModelCheckpoint

from tensorflow.keras.models import load_model# carregar um modelo já treinado (.h5)
from tensorflow.keras.models import model_from_json # recriar a arquitetura a partir do json salvo
from tensorflow.keras.applications import VGG16 # o modelo pré-treinado que vamos reaproveitar

## Etapa 6 - Dividir em conjuntos para treinamento e validação

In [ ]:
# separa 10% dos dados pra teste
x_train, x_test, y_train, y_test = train_test_split(faces, emocoes, test_size=0.1, random_state=42)
# do que sobrou, separa mais 10% pra validação (usada durante o treinamento)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=41)

print("Número de imagens no conjunto de treinamento:", len(x_train))
print("Número de imagens no conjunto de testes:", len(x_test))
print("Número de imagens no conjunto de validação:", len(y_val))

In [ ]:
# salva o conjunto de teste em disco, pra depois recarregar sem refazer o pré-processamento
np.save('mod_xtest', x_test)
np.save('mod_ytest', y_test)

## Etapa 7 - Arquitetura do Modelo (CNN)

### Arquitetura do modelo

Vamos utilizar o modelo da VGG-16, este que possui pesos treinados com o dataset ImageNet, que é um conjunto de dados composto por mais de 14 milhões de imagens classificadas em 1000 classes.

![alt text](https://drive.google.com/uc?id=1vFKAnNPuQ_eHxeaoscwRtB5P2Gg1kyhM)

**Arquitetura da VGG-16** ([créditos da imagem](https://towardsdatascience.com/step-by-step-vgg16-implementation-in-keras-for-beginners-a833c686ae6c))

In [ ]:
num_classes = 7
width, height = 48, 48
batch_size = 16
epochs = 30

# carrega a VGG-16 já treinada no ImageNet, sem a "cabeça" de classificação original (include_top=False)
# porque vamos trocar essa parte final por uma nova, feita pra classificar as 7 emoções
vgg = VGG16(input_shape=(width, height, 3), weights='imagenet', include_top=False)

In [ ]:
vgg.trainable=False # congela os pesos da VGG-16: eles não serão atualizados durante o treino
global_average_layer = GlobalAveragePooling2D() # resume cada mapa de características num único valor
prediction_layer = Dense(num_classes,activation='softmax') # nova camada de saída, com as 7 emoções

* É importante congelar as camadas convolucionais antes de compilar e treinar o modelo.
* Ao congelar ou definir layer.trainable = False, você evita que os pesos em uma determinada camada sejam atualizados durante o treinamento.

In [ ]:
# empilha: VGG-16 (congelada) -> global average pooling -> nova camada de classificação
model = Sequential([
  vgg,
  global_average_layer,
  prediction_layer
])

(para maiores explicações do porquê fizemos isso leia o texto mais abaixo)

In [ ]:
model.summary()

Essa é apenas uma das possibilidades de se usar VGG-16 para transfer learning. A maneira de se utilizar transfer learning vai depender da situação. Podemos definir 4 cenários mais comuns:

* Conjunto de dados pequeno e diferente do modelo pré-treinado
* Conjunto de dados grande e diferente do modelo pré-treinado
* Conjunto de dados grande e similar ao do modelo pré-treinado
* Conjunto de dados pequeno e similar ao modelo pré-treinado

Se usarmos somente a base fer2013 então nos enquadramos mais no segundo cenário, pois é considerada uma boa quantidade de imagens (embora tenha casos onde se tenha bem mais, porém 30.000~ é um número grande em comparação a maioria das situações) e tem bastante imagens parecidas mas no geral mesmo são diferentes pois o modelo pré-treinado utilizou imagens muito diversas para o treinamento.

Mas vamos pensar assim: comparando com o conjunto do imageNet (milhões de imagens) nosso conjunto de imagens é bem pequeno. Outro ponto: como queremos aproveitar desse modelo pré-treinado, vamos supor que nossas imagens sejam mais parecidas, caso fossem imagens muito diferentes não faria tanto sentido utilizar um modelo pré-treinado, seria melhor criar a própria CNN como fizemos em exemplos anteriores (claro que, desconsiderando se as imagens treinadas são muito parecidas ou não, outra vantagem importante em relação ao uso de modelos pré-treinados é a maior velocidade no tempo de treinamento, que leva menos). Portanto, vamos seguir nesse exemplo as recomendações do quarto cenário, que são:

* Adicionar camadas densamente conectas depois das camadas convolucionais;
* Congelar os pesos das primeiras camadas convolucionais ou todas camadas convolucionais.

Foi isso que fizemos mais acima no código, com `vgg.trainable=False ` (para congelar as camadas)

As demais recomendações (e sugestões de qual abordagem seguir para os outros 3 cenários) podem ser vistas com mais detalhe nesse artigo: https://medium.com/ensina-ai/tutorial-transfer-learning-3972cac5e9b5

Você pode fazer o treinamento seguindo as sugestões para outros cenários, assim é possível ter diferentes resultados (melhores ou piores, vai depender das condições comentadas).

## Etapa 8 - Compilando o modelo

In [ ]:
model.compile(loss=categorical_crossentropy,    # loss padrão pra classificação com várias classes
              optimizer=Adam(lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-7),
              metrics=['accuracy'])
arquivo_modelo = diretorio + "modelo_vgg_expressoes.h5" # arquivo do modelo
arquivo_modelo_json = diretorio + "modelo_vgg_expressoes.json" # arquivo do json, para salvar a arquitetura
# reduz o learning rate quando a loss de validação para de melhorar
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.9, patience=3, verbose=1)
# para o treinamento mais cedo se não houver melhora (evita overfitting)
early_stopper = EarlyStopping(monitor='val_loss', min_delta=0, patience=8, verbose=1, mode='auto')
# salva automaticamente o melhor modelo (menor val_loss) durante o treino
checkpointer = ModelCheckpoint(arquivo_modelo, monitor='val_loss', verbose=1, save_best_only=True)

### Salvando a arquitetura do modelo em um arquivo JSON

In [ ]:
# salva só a arquitetura do modelo (as camadas) num arquivo json
# os pesos ficam separados, no .h5 salvo pelo checkpointer durante o treino
model_json = model.to_json()
with open(arquivo_modelo_json, "w") as json_file:
    json_file.write(model_json)

## Etapa 9 - Treinando o modelo

In [ ]:
# treina o modelo
# history guarda o histórico de loss/acurácia de cada epoch, usamos depois pra plotar o gráfico
history = model.fit(np.array(x_train), np.array(y_train),
          batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(np.array(x_val), np.array(y_val)),
          shuffle=True, # embaralha os dados a cada epoch
          callbacks=[lr_reducer, early_stopper, checkpointer])

Dica de como você pode melhorar:
Primeiramente recomendamos se aprofundar mais no estudo sobre arquitetura do VGG-16 (ou outro modelo pré-treinado que você quer utilizar), assim será possível ajustar seu modelo ao melhor caso e os melhores parâmetros, dependendo do seu conjunto de imagens (seja para detecção de emoções ou outro tipo de classificação).

Para ter melhores resultados com o VGG-16 podemos utilizar o que é chamado de **bottleneck features**, que são basicamente as características finais da imagem que obtemos no final da rede neural convolucional.

Mais sobre bottleneck features: https://ai.stackexchange.com/questions/3089/what-are-bottleneck-features

https://www.quora.com/What-is-the-definition-of-bottleneck-features-in-transfer-learning

Após gerar os bottleneck features para cada uma das imagens do conjunto de treinamentos (model.predict() do VGG-16) é possível armazenar esses valores em uma array numpy.

Esse tutorial explica detalhadamente como é possível fazer isso: https://www.kaggle.com/payat123/vgg16-bottleneck-features-to-predict-using-keras

Assim a tendencia é que seus resultados sejam bem melhores pois a rede vai ser treinada de forma mais adequada.


Outras arquiteturas pré-treinadas:
* Xception
* VGG-19
* ResNet, ResNetV2, ResNeXt
* InceptionV3
* MobileNet, MobileNetV2
* DenseNet
* NASNet
* etc.



Caso seu conjunto de dados seja bem pequeno então você pode tirar mais proveito do modelo pré-treinado, pois uma das principais vantagens de utilizar essa arquitetura (além de consumir menos tempo no treinamento) é mais notável em situações onde há poucas imagens para treinamento.

## Gerando gráfico da melhora em cada etapa do treinamento

In [ ]:
# plota dois gráficos lado a lado: acurácia e loss (treino x validação) ao longo das epochs
def plota_historico_modelo(historico_modelo):
    fig, axs = plt.subplots(1,2,figsize=(15,5))
    axs[0].plot(range(1,len(historico_modelo.history['accuracy'])+1),
                historico_modelo.history['accuracy'],'r')
    axs[0].plot(range(1,len(historico_modelo.history['val_accuracy'])+1),
                historico_modelo.history['val_accuracy'],'b')
    axs[0].set_title('Acurácia do Modelo')
    axs[0].set_ylabel('Acuracia')
    axs[0].set_xlabel('Epoch')
    axs[0].set_xticks(np.arange(1,len(historico_modelo.history['accuracy'])+1),
                      len(historico_modelo.history['accuracy'])/10)
    axs[0].legend(['training accuracy', 'validation accuracy'], loc='best')

    axs[1].plot(range(1,len(historico_modelo.history['loss'])+1),
                historico_modelo.history['loss'],'r')
    axs[1].plot(range(1,len(historico_modelo.history['val_loss'])+1),
                historico_modelo.history['val_loss'],'b')
    axs[1].set_title('Perda/Loss do Modelo')
    axs[1].set_ylabel('Loss')
    axs[1].set_xlabel('Epoch')
    axs[1].set_xticks(np.arange(1,len(historico_modelo.history['loss'])+1),
                      len(historico_modelo.history['loss'])/10)
    axs[1].legend(['training loss', 'validation Loss'], loc='best')
    fig.savefig('historico_modelo_mod01.png')  # salva a imagem do gráfico em disco
    plt.show()

plota_historico_modelo(history)

### Verificando a acurácia do modelo

In [ ]:
# avalia o modelo já treinado usando o conjunto de teste (imagens que ele nunca viu)
scores = model.evaluate(np.array(x_test), np.array(y_test), batch_size=batch_size)
print("Acurácia: " + str(scores[1]))
print("Perda/Loss: " + str(scores[0]))

## Carregaremos os dados para gerar a matriz de confusão

In [ ]:
true_y=[]  # vai guardar o índice da emoção correta (rótulo verdadeiro)
pred_y=[]  # vai guardar o índice da emoção que o modelo previu

# recarrega o conjunto de teste que salvamos antes
x = np.load('mod_xtest.npy')
y = np.load('mod_ytest.npy')

# recarrega o modelo a partir da arquitetura (json) + pesos (h5)
json_file = open(arquivo_modelo_json, 'r')
loaded_model_json = json_file.read()
json_file.close()
loaded_model = model_from_json(loaded_model_json)
loaded_model.load_weights(arquivo_modelo)

y_pred= loaded_model.predict(x)  # faz a previsão pra todo o conjunto de teste

yp = y_pred.tolist()
yt = y.tolist()
count = 0
for i in range(len(y)):
    yy = max(yp[i]) # maior probabilidade prevista
    yyt = max(yt[i]) # valor "1" do one-hot (rótulo verdadeiro)
    pred_y.append(yp[i].index(yy))# índice da emoção prevista
    true_y.append(yt[i].index(yyt)) # índice da emoção correta
    if(yp[i].index(yy)== yt[i].index(yyt)):
        count+=1 # conta quantas vezes acertou
acc = (count/len(y))*100

# salva os resultados (usados na matriz de confusão)
np.save('truey__mod01', true_y)
np.save('predy__mod01', pred_y)
print("Acurácia no conjunto de testes: "+str(acc)+"%")

## Gerando a Matriz de Confusão

In [ ]:
from sklearn.metrics import confusion_matrix

y_true = np.load('truey__mod01.npy')
y_pred = np.load('predy__mod01.npy')

cm = confusion_matrix(y_true, y_pred)  # monta a matriz comparando o rótulo real x o previsto
expressoes = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]
titulo='Matriz de Confusão'
print(cm)

In [ ]:
# desenha a matriz de confusão como um "mapa de calor"
import itertools
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title(titulo)
plt.colorbar()
tick_marks = np.arange(len(expressoes))
plt.xticks(tick_marks, expressoes, rotation=45)
plt.yticks(tick_marks, expressoes)
fmt = 'd'
thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    # escreve o número dentro de cada célula da matriz
    plt.text(j, i, format(cm[i, j], fmt),
            horizontalalignment="center",
            color="white" if cm[i, j] > thresh else "black")

plt.ylabel('Classificação Correta')
plt.xlabel('Predição')
plt.savefig('matriz_confusao_mod01.png')
plt.show()

## Testando brevemente o modelo

In [ ]:
# carrega uma foto de teste, diferente das usadas no treinamento
imagem = cv2.imread(diretorio + "testes/teste02.jpg")
cv2_imshow(imagem)

In [ ]:
# recarrega o melhor checkpoint salvo durante o treino, pra rodar o teste rápido abaixo
model = load_model(diretorio + "modelo_vgg_expressoes.h5")
scores = model.evaluate(np.array(x_test), np.array(y_test), batch_size=batch_size)
print("Perda/Loss: " + str(scores[0]))
print("Acurácia: " + str(scores[1]))

**Atenção:** para reconhecer as emoções usando essa arquitetura nós precisamos fazer duas pequenas mudanças no código:
* primeiro, após converter para grayscale precisamos converter novamente para RGB
* também não usaremos o np.expand_dims(, -1) pois não se faz necessário já que o shape dessa vez já possui mais uma dimensão (3 canais de cor, diferente dos outros exemplos que era grayscale e usávamos uma dimensão a menos porque a imagem não tinha os canais de cores)

In [ ]:
expressoes = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]
original = imagem.copy()
gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
face_cascade = cv2.CascadeClassifier(diretorio + 'haarcascade_frontalface_default.xml')
faces = face_cascade.detectMultiScale(gray, 1.1, 3)
for (x, y, w, h) in faces:
    cv2.rectangle(original, (x, y), (x + w, y + h), (0, 255, 0), 1)  # desenha o retângulo em volta do rosto
    roi_gray = gray[y:y + h, x:x + w] # extrai só a região do rosto (ROI)
    roi_gray = cv2.cvtColor(roi_gray, cv2.COLOR_GRAY2RGB)# converte de volta pra 3 canais (RGB), já que a VGG-16 espera imagem colorida
    roi_gray = roi_gray.astype("float") / 255.0 # normaliza
    # também não usaremos o np.expand_dims(, -1) porque a imagem já tem os 3 canais de cor
    cropped_img = np.expand_dims(cv2.resize(roi_gray, (48, 48)), 0)
    cv2.normalize(cropped_img, cropped_img, alpha=0, beta=1,
                  norm_type=cv2.NORM_L2, dtype=cv2.CV_32F)
    prediction = model.predict(cropped_img)[0] # prediz a emoção
    cv2.putText(original, expressoes[int(np.argmax(prediction))], (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)  # escreve a emoção acima do rosto
cv2_imshow(original)